# 2-1-3 데이터 전처리
교과서 **66~73쪽**

지난 시간에는 파일을 열어 **편향**을 봤습니다.  
오늘은 두 표를 합친 뒤, 빈칸과 튀는 값을 손질하고 **종류를 잘 가리는 속성**을 고릅니다.

| 오늘 할 일 | 왜 필요한가? |
|---|---|
| 속성을 구분하고, 합치고, 결측·이상치를 처리한 뒤 핵심 속성을 고르기 | 손질하지 않은 표로는 모델이 제대로 배우지 못합니다 |

**준비물:** 같은 폴더의 `Iris1.csv`, `Iris2.csv`  
**실행:** 위에서부터 ▶. 에러가 나면 위 칸을 건너뛰었는지 봅니다.


## 0. 도구 꺼내기 · 파일 열기

한글 CSV는 `encoding='cp949'`로 엽니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import platform

if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
# 코랩에서 한글이 깨지면:  !pip install -q koreanize-matplotlib  후  import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False

df1 = pd.read_csv('Iris1.csv', encoding='cp949')
df2 = pd.read_csv('Iris2.csv', encoding='cp949')
print('Iris1:', df1.shape, ' / Iris2:', df2.shape)
df1.head()


> 코랩이면 먼저 `from google.colab import files` → `files.upload()` 로 두 CSV를 올리세요.


---

# 1. 열(속성)을 어떻게 구분할까? (66쪽)

| 구분 | 뜻 | 이 표에서 |
|---|---|---|
| 수치형 | 크기 비교·평균이 되는 숫자 | 꽃받침·꽃잎의 길이와 너비 |
| 범주형 | 이름·항목 | 종류 |
| 독립 변수 | 맞힐 때 보는 입력 | 네 가지 길이·너비 |
| 종속 변수 | 맞히고 싶은 정답 | 종류 (분류 문제) |
| 식별자 | 행 번호. 꽃의 특징 아님 | 일련번호 |


In [ ]:
print('열 이름:', df1.columns.tolist())
print()
print(df1.dtypes)


> **생각하기 1**
> `일련번호`를 학습에 넣으면 안 되는 이유는? (1~50번이 세토사처럼, 번호가 종류와 함께 커집니다.)
> →


---

# 2. 두 표를 하나로 합치기 (67쪽)

`Iris2.csv`는 종류별 10개씩 더한 최근 측정입니다. 빈칸이 있어도 지금은 그대로 붙입니다.


In [ ]:
print('Iris2 앞부분 — NaN(빈칸)이 보이면 정상입니다.')
df2.head(8)


In [ ]:
df = pd.concat([df1, df2], ignore_index=True)
print('합친 뒤:', df.shape, '(150 + 30)')
print(df['종류'].value_counts())


> **생각하기 2**
> 행을 늘리면 무엇이 좋아지나요? 새로 넣은 값에 빈칸·잘못된 측정이 있으면?
> →


---

# 3. 결측치 확인하고 처리하기 (68쪽)

결측치는 비어 있는 칸(`NaN`)입니다. 처리 방법은 세 가지입니다.

| | 방법 | 이런 때 |
|---|---|---|
| (가) 열 삭제 | 그 속성 전체를 버림 | 한 열이 거의 다 비어 있을 때 |
| (나) 값 대체 | 평균 등으로 채움 | 빈칸이 아주 적을 때. 가짜 값이 들어감 |
| (다) 행 삭제 | 빈칸 있는 행만 지움 | 전체 행이 충분할 때 ← **교과서가 선택한 방법** |


In [ ]:
print(df.isna().sum())
빈행 = df[df.isna().any(axis=1)]
print('결측이 있는 행 수:', len(빈행))
빈행


In [ ]:
df3 = df.dropna(axis=0)
print('행 삭제 후:', len(df), '→', len(df3))
print(df3['종류'].value_counts())


> **생각하기 3**
> 지금은 왜 (가) 열 삭제가 과할까요? 전체 평균으로 꽃잎 길이를 채우면 어떤 문제가 생길까요?
> →


---

# 4. 이상치 확인하고 처리하기 (69~71쪽)

이상치는 덩어리에서 많이 벗어난 값입니다.  
박스 플롯에서 상자 밖 점을 이상치로 봅니다. (울타리 = Q1 − 1.5×IQR, Q3 + 1.5×IQR)


In [ ]:
print('종류별 꽃받침 길이 — max 순서가 mean 순서와 다르면 이상치를 의심합니다.')
display(df3.groupby('종류')['꽃받침 길이'].describe().round(2))

속성들 = ['꽃받침 길이', '꽃받침 너비', '꽃잎 길이', '꽃잎 너비']
for 속성 in 속성들:
    sns.catplot(data=df3, x='종류', y=속성, kind='box', height=3.2, aspect=1.05)
    plt.title(속성)
    plt.show()


> **생각하기 4**
> 상자 밖으로 점이 찍힌 종류·속성을 적으세요. 평균 순서는 버지니카 > 버시컬러인데, 최댓값은 어떤가요?
> →


In [ ]:
def 이상치_행번호(data, 종류이름, 속성):
    부분 = data[data['종류'] == 종류이름][속성]
    q1, q3 = 부분.quantile(0.25), 부분.quantile(0.75)
    iqr = q3 - q1
    조건 = (data['종류'] == 종류이름) & (
        (data[속성] < q1 - 1.5 * iqr) | (data[속성] > q3 + 1.5 * iqr)
    )
    return data[조건].index.tolist()

지울 = []
for 종류이름 in df3['종류'].unique():
    for 속성 in 속성들:
        지울.extend(이상치_행번호(df3, 종류이름, 속성))
지울 = sorted(set(지울))
print('지울 행:', 지울)
display(df3.loc[지울, ['일련번호'] + 속성들 + ['종류']])

df4 = df3.drop(index=지울)
print('이상치 제거 후:', len(df3), '→', len(df4))


---

# 5. 핵심 속성 고르기 (72쪽)

세 종류가 잘 나뉘는 속성을 고릅니다. 산점도로 눈으로, 히트맵으로 숫자로 봅니다.


In [ ]:
sns.pairplot(df4.drop(columns=['일련번호']), hue='종류', markers=['o', 's', 'D'])
plt.show()

수치열 = ['꽃받침 길이', '꽃받침 너비', '꽃잎 길이', '꽃잎 너비']
상관 = df4[수치열].corr(method='pearson')
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(상관, annot=True, fmt='.2f', cmap='RdYlGn_r')
plt.title('속성 사이 상관계수')
plt.tight_layout()
plt.show()


> **생각하기 5**
> 세 종류가 가장 또렷이 나뉜 조합은 무엇인가요? (교과서: 꽃잎 길이·꽃잎 너비, 상관 약 0.96)
> →


---

# 오늘 정리

| 확인 항목 | 내가 본 결과 | 그래서 어떻게 했는가? |
|---|---|---|
| 일련번호의 역할 |  |  |
| 결측치 개수·처리 |  |  |
| 이상치를 어떻게 봤는가 |  |  |
| 핵심 속성 |  |  |

**자기 점검**

- [ ] 수치형/범주형, 독립/종속 변수를 구분할 수 있다.
- [ ] `concat`으로 두 표를 합치고 `isna().sum()`으로 빈칸을 확인했다.
- [ ] 결측은 행 삭제로 처리한 이유를 말할 수 있다.
- [ ] 박스 플롯·IQR로 이상치를 찾고, 산점도·히트맵으로 핵심 속성을 골랐다.

다음 시간(`2-1-4`)에는 **어떤 유형·알고리즘으로 종류를 맞힐지** 정합니다.  
(교과서 73쪽 `tips` 연습은 시간이 남으면 같은 순서로 해 보세요.)
